In [3]:
"""
=============================================================================
CHEQUEO DE CALIDAD — PANEL MAESTRO (con visualizaciones)
=============================================================================
Input:  Data/Processed/panel_maestro.parquet
Output: Data/Processed/Figuras/diagnostico_visual_panel.pdf  (reporte completo)
        Data/Processed/Figuras/*.png                          (cada gráfico suelto)
        Data/Processed/Figuras/resumen_calidad.txt             (resumen en texto)
Dependencies: pip install pandas numpy matplotlib pyarrow
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings("ignore")

# =============================================================================
# 0. PATHS & CONFIG
# =============================================================================
ROOT = Path(r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data")
PROCESSED = ROOT / "Processed"
FIGURES = PROCESSED / "Figuras"
FIGURES.mkdir(parents=True, exist_ok=True)

PANEL_PATH = PROCESSED / "panel_maestro.parquet"
TARGET = "TASA_DESERCION_MPIO"

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

pdf = PdfPages(FIGURES / "diagnostico_visual_panel.pdf")
summary_lines = []


def log(msg):
    """Imprime en consola y guarda en el resumen de texto."""
    print(msg)
    summary_lines.append(msg)


def save_fig(fig, name):
    """Guarda la figura como PNG suelto y la agrega al PDF consolidado."""
    fig.tight_layout()
    fig.savefig(FIGURES / f"{name}.png", dpi=150, bbox_inches="tight")
    pdf.savefig(fig)
    plt.close(fig)


# =============================================================================
# 1. CARGA DEL PANEL
# =============================================================================
log("\n>>> MODULE 1: CARGANDO PANEL MAESTRO")

if not PANEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró: {PANEL_PATH}")

panel = pd.read_parquet(PANEL_PATH)

log(f"  Filas: {len(panel):,} | Columnas: {panel.shape[1]}")
log(f"  Años: {sorted(panel['PERIODO_ANIO'].dropna().unique().tolist())}")
log(f"  Sedes únicas: {panel['SEDE_CODIGO'].nunique():,}")
if "SAMPLE" in panel.columns:
    log(f"  TRAIN/TEST: {panel['SAMPLE'].value_counts().to_dict()}")

# =============================================================================
# 2. COMPLETITUD — TOP 30 VARIABLES CON MÁS NULOS
# =============================================================================
log("\n>>> MODULE 2: COMPLETITUD")

pct_nulos = (panel.isnull().mean() * 100).sort_values(ascending=False)
top_nulos = pct_nulos.head(30)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_nulos.index[::-1], top_nulos.values[::-1], color="firebrick")
ax.set_xlabel("% de valores nulos")
ax.set_title("Top 30 variables con más nulos")
ax.axvline(20, color="black", linestyle="--", linewidth=1, label="Umbral 20%")
ax.legend()
save_fig(fig, "01_completitud_top30")

vars_criticas = pct_nulos[pct_nulos > 20]
log(f"  Variables con >20% nulos: {len(vars_criticas)} de {panel.shape[1]}")

# =============================================================================
# 3. DISTRIBUCIÓN DEL TARGET
# =============================================================================
log("\n>>> MODULE 3: DISTRIBUCIÓN DEL TARGET")

if TARGET in panel.columns:
    target_data = panel[TARGET].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(target_data, bins=50, color="steelblue", edgecolor="white")
    axes[0].set_title(f"Distribución de {TARGET}")
    axes[0].set_xlabel(TARGET)
    axes[0].set_ylabel("Frecuencia (sede-año)")

    axes[1].boxplot(target_data, vert=True)
    axes[1].set_title(f"Boxplot de {TARGET}")
    axes[1].set_xticklabels([TARGET])
    save_fig(fig, "02_distribucion_target")

    log(f"  {TARGET}: media={target_data.mean():.4f} | mediana={target_data.median():.4f} | "
        f"std={target_data.std():.4f} | cobertura={target_data.notna().mean()*100:.1f}%")
else:
    log(f"  [WARN] No se encontró la columna target '{TARGET}'")

# =============================================================================
# 4. EVOLUCIÓN TEMPORAL — TARGET Y MATRÍCULA POR AÑO
# =============================================================================
log("\n>>> MODULE 4: EVOLUCIÓN TEMPORAL")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if TARGET in panel.columns:
    target_by_year = panel.groupby("PERIODO_ANIO")[TARGET].mean()
    axes[0].plot(target_by_year.index, target_by_year.values, marker="o", color="darkred")
    axes[0].set_title(f"{TARGET} promedio por año")
    axes[0].set_xlabel("Año")
    axes[0].set_ylabel(TARGET)
    if "FLAG_PANDEMIA" in panel.columns:
        axes[0].axvline(2020, color="gray", linestyle="--", alpha=0.6, label="Pandemia (2020)")
        axes[0].legend()

if "MATRICULA_TOTAL" in panel.columns:
    matricula_by_year = panel.groupby("PERIODO_ANIO")["MATRICULA_TOTAL"].sum()
    axes[1].plot(matricula_by_year.index, matricula_by_year.values, marker="o", color="steelblue")
    axes[1].set_title("Matrícula total por año")
    axes[1].set_xlabel("Año")
    axes[1].set_ylabel("Matrícula total")

save_fig(fig, "03_evolucion_temporal")

# =============================================================================
# 5. TOP CORRELACIONES CON EL TARGET
# =============================================================================
log("\n>>> MODULE 5: CORRELACIONES CON EL TARGET")

if TARGET in panel.columns:
    num_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != TARGET and int(panel[c].notna().sum()) > 500]

    corr = (
        panel[num_cols + [TARGET]]
        .corr()[TARGET]
        .drop(TARGET, errors="ignore")
        .dropna()
        .sort_values(key=abs, ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ["firebrick" if v < 0 else "steelblue" for v in corr.values]
    ax.barh(corr.index[::-1], corr.values[::-1], color=colors[::-1])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Correlación de Pearson")
    ax.set_title(f"Top 15 variables correlacionadas con {TARGET}")
    save_fig(fig, "04_top_correlaciones")

    log("  Top 5 correlaciones:")
    for var, val in corr.head(5).items():
        log(f"    {var}: {val:.4f}")

# =============================================================================
# 6. MATRIZ DE CORRELACIÓN — VARIABLES CLAVE
# =============================================================================
log("\n>>> MODULE 6: MATRIZ DE CORRELACIÓN (variables clave)")

vars_clave = [
    TARGET, "TASA_REPITENCIA_MPIO", "DESERCION_LAG1", "REPITENCIA_LAG1",
    "DESERCION_MA2_LAG", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO",
    "INDICE_VULNERABILIDAD", "IDX_FEMINIDAD", "IPM_IPM",
    "ICFES_PROM_PUNT_GLOBAL", "FLAG_PDET", "FLAG_ZOMAC",
]
vars_clave = [c for c in vars_clave if c in panel.columns]

if len(vars_clave) >= 2:
    corr_matrix = panel[vars_clave].corr()

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(vars_clave)))
    ax.set_yticks(range(len(vars_clave)))
    ax.set_xticklabels(vars_clave, rotation=90)
    ax.set_yticklabels(vars_clave)
    for i in range(len(vars_clave)):
        for j in range(len(vars_clave)):
            val = corr_matrix.values[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=7, color="white" if abs(val) > 0.5 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8, label="Correlación")
    ax.set_title("Matriz de correlación — variables clave")
    save_fig(fig, "05_matriz_correlacion")

# =============================================================================
# 7. COBERTURA POR AÑO — VARIABLES CLAVE
# =============================================================================
log("\n>>> MODULE 7: COBERTURA POR AÑO")

cov_vars = [
    "MATRICULA_TOTAL", TARGET, "TASA_REPITENCIA_MPIO", "DESERCION_LAG1",
    "IPM_IPM", "FLAG_PDET", "FLAG_ZOMAC", "ICFES_PROM_PUNT_GLOBAL",
]
cov_vars = [c for c in cov_vars if c in panel.columns]

if cov_vars:
    cov_by_year = panel.groupby("PERIODO_ANIO")[cov_vars].apply(lambda x: x.notna().mean() * 100)

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(cov_by_year.values, cmap="YlGnBu", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(cov_vars)))
    ax.set_xticklabels(cov_vars, rotation=45, ha="right")
    ax.set_yticks(range(len(cov_by_year)))
    ax.set_yticklabels(cov_by_year.index.astype(int))
    for i in range(cov_by_year.shape[0]):
        for j in range(cov_by_year.shape[1]):
            val = cov_by_year.values[i, j]
            ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                    fontsize=8, color="white" if val > 50 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8, label="% cobertura")
    ax.set_title("Cobertura (%) por año y variable")
    save_fig(fig, "06_cobertura_por_anio")

# =============================================================================
# 8. TRAIN VS TEST — DISTRIBUCIÓN DEL TARGET
# =============================================================================
log("\n>>> MODULE 8: TRAIN VS TEST")

if "SAMPLE" in panel.columns and TARGET in panel.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    for sample_val, color in [("TRAIN", "steelblue"), ("TEST", "firebrick")]:
        data = panel.loc[panel["SAMPLE"] == sample_val, TARGET].dropna()
        if len(data) > 0:
            ax.hist(data, bins=40, alpha=0.5, label=sample_val, color=color, density=True)
    ax.set_title(f"Distribución de {TARGET}: TRAIN vs TEST")
    ax.set_xlabel(TARGET)
    ax.set_ylabel("Densidad")
    ax.legend()
    save_fig(fig, "07_train_vs_test")

# =============================================================================
# 9. TARGET SEGÚN FLAGS (PDET / ZOMAC / PANDEMIA)
# =============================================================================
log("\n>>> MODULE 9: TARGET SEGÚN FLAGS")

flag_vars = [c for c in ["FLAG_PDET", "FLAG_ZOMAC", "FLAG_PANDEMIA", "FLAG_DECLIVE_MATRICULA"]
             if c in panel.columns]

if flag_vars and TARGET in panel.columns:
    fig, axes = plt.subplots(1, len(flag_vars), figsize=(4.5 * len(flag_vars), 5), squeeze=False)
    axes = axes[0]

    for ax, flag in zip(axes, flag_vars):
        grupo_0 = panel.loc[panel[flag] == 0, TARGET].dropna()
        grupo_1 = panel.loc[panel[flag] == 1, TARGET].dropna()
        ax.boxplot([grupo_0, grupo_1], label=["0", "1"])
        ax.set_title(flag)
        ax.set_xlabel(flag)

    fig.suptitle(f"{TARGET} según flags binarios")
    save_fig(fig, "08_target_por_flags")

# =============================================================================
# 10. TARGET POR REGIÓN
# =============================================================================
log("\n>>> MODULE 10: TARGET POR REGIÓN")

if "REGION_DANE" in panel.columns and TARGET in panel.columns:
    regiones = panel["REGION_DANE"].dropna().unique().tolist()
    data_por_region = [panel.loc[panel["REGION_DANE"] == r, TARGET].dropna() for r in regiones]

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.boxplot(data_por_region, label=regiones)
    ax.set_title(f"{TARGET} por región DANE")
    ax.set_ylabel(TARGET)
    plt.xticks(rotation=30, ha="right")
    save_fig(fig, "09_target_por_region")

# =============================================================================
# 11. AUTOCORRELACIÓN LAG-1
# =============================================================================
log("\n>>> MODULE 11: AUTOCORRELACIÓN LAG-1")

if "DESERCION_LAG1" in panel.columns and TARGET in panel.columns:
    lag_data = panel[[TARGET, "DESERCION_LAG1"]].dropna()

    if len(lag_data) > 0:
        r = lag_data.corr().loc[TARGET, "DESERCION_LAG1"]

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.scatter(lag_data["DESERCION_LAG1"], lag_data[TARGET], alpha=0.15, s=10, color="steelblue")
        lims = [lag_data.min().min(), lag_data.max().max()]
        ax.plot(lims, lims, color="black", linestyle="--", linewidth=1, label="y = x")
        ax.set_xlabel("DESERCION_LAG1 (año anterior)")
        ax.set_ylabel(TARGET)
        ax.set_title(f"Autocorrelación lag-1 (R={r:.3f}, R²={r**2:.3f})")
        ax.legend()
        save_fig(fig, "10_autocorrelacion_lag1")

        log(f"  R={r:.4f} | R²={r**2:.4f}")

# =============================================================================
# 12. CIERRE
# =============================================================================
pdf.close()

resumen_path = FIGURES / "resumen_calidad.txt"
resumen_path.write_text("\n".join(summary_lines), encoding="utf-8")

log("\n>>> CHEQUEO DE CALIDAD COMPLETO")
log(f"  PDF:     {FIGURES / 'diagnostico_visual_panel.pdf'}")
log(f"  PNGs:    {FIGURES}")
log(f"  Resumen: {resumen_path}")



>>> MODULE 1: CARGANDO PANEL MAESTRO
  Filas: 319,609 | Columnas: 70
  Años: [2018, 2019, 2020, 2021, 2022, 2023]
  Sedes únicas: 56,557
  TRAIN/TEST: {'TRAIN': 266463, 'TEST': 53146}

>>> MODULE 2: COMPLETITUD
  Variables con >20% nulos: 34 de 70

>>> MODULE 3: DISTRIBUCIÓN DEL TARGET
  TASA_DESERCION_MPIO: media=0.0353 | mediana=0.0324 | std=0.0188 | cobertura=100.0%

>>> MODULE 4: EVOLUCIÓN TEMPORAL

>>> MODULE 5: CORRELACIONES CON EL TARGET
  Top 5 correlaciones:
    DESERCION_MA2_LAG: 0.5996
    DESERCION_LAG1: 0.5474
    FLAG_PDET: 0.2795
    IPM_ALFABETISMO: -0.2694
    IPM_EMPLEO_FORMAL: -0.2693

>>> MODULE 6: MATRIZ DE CORRELACIÓN (variables clave)

>>> MODULE 7: COBERTURA POR AÑO

>>> MODULE 8: TRAIN VS TEST

>>> MODULE 9: TARGET SEGÚN FLAGS

>>> MODULE 10: TARGET POR REGIÓN

>>> MODULE 11: AUTOCORRELACIÓN LAG-1
  R=0.5474 | R²=0.2997

>>> CHEQUEO DE CALIDAD COMPLETO
  PDF:     C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data\Proces